In [64]:
import numpy as np
import pandas as pd
import openmatrix as omx
from pathlib import Path
from dbfread import DBF
import copy
from datetime import datetime
import os
import geopandas as gpd
from libpysal.weights import KNN


In [65]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


## Inputs


In [66]:
path_inputs = Path.cwd() / "inputs"
path_outputs = Path.cwd() / "outputs"
path_start_files = path_inputs / "input_files/C28"

path_coeff_file = path_inputs / "coefficients_cal8.block"
path_se_file = path_start_files / "SE_File.dbf"
path_pa_file = path_start_files / "pa_initial.dbf"
path_tel_file = path_start_files / "telecommute.dbf"
path_hh_files = [
    path_start_files / f"HH{i}_PercTrips_segment_hbw.dbf" for i in range(1, 7)
]
path_taz_file = path_start_files / "WFv1000_TAZ.dbf"
path_taz_shpfile = path_start_files / "WFv1000_TAZ.shp"
path_dens_file = path_start_files / "IntersectionDensity.dbf"

# Skims and Logsums (assuming they are already converted to OMX in this folder)
path_skm_hwy = path_start_files / "skm_auto_Pk.omx"
path_skm_walk = path_start_files / "Best_Walk_Skims.omx"
path_logsums = path_start_files / "HBW_logsums_Pk.omx"
path_skm_transit_pk = path_start_files / "1Skm_TotTransitTime_Pk.omx"
path_skm_transit_ok = path_start_files / "1Skm_TotTransitTime_Ok.omx"

# We load this once here just to get the segment names for loading matrices
# initial_coeffs = load_block_coefficients(path_coeff_file)
# segments = list(initial_coeffs.keys())


In [67]:
used_zones = 3629
dummy_zones_str = "3563-3600"
external_zones_str = "3601-3629"

# read dbf files
df_se = pd.DataFrame(iter(DBF(path_se_file)))
df_pa = pd.DataFrame(iter(DBF(path_pa_file)))
df_tel = pd.DataFrame(iter(DBF(path_tel_file)))
df_hh = [pd.DataFrame(iter(DBF(f))) for f in path_hh_files]
df_taz = pd.DataFrame(iter(DBF(path_taz_file)))
gdf_taz = gpd.read_file(path_taz_shpfile)
df_dens = pd.DataFrame(iter(DBF(path_dens_file)))

# Create dummy/external mask
mask_external_dummys = np.zeros(used_zones, dtype=bool)
# apply_range_mask(mask_external_dummys, dummy_zones_str)
# apply_range_mask(mask_external_dummys, external_zones_str)


In [68]:
hts_hh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.hh"
).to_dataframe()
hts_person_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.person"
).to_dataframe()
hts_trip_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.trip_linked"
).to_dataframe()
hts_veh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.vehicle"
).to_dataframe()


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


# Estimation Process 1


## Observed Data


In [ ]:
hts_trip_23_merge = hts_trip_23.copy()
hts_trip_23_merge = hts_trip_23_merge[
    [
        "unique_id",
        "hh_id",
        "person_id",
        "vehicle_id",
        "pCO_TAZID_USTMv4",
        "aCO_TAZID_USTMv4",
        "pSUBAREAID",
        "aSUBAREAID",
        "PURP7_t",
        "trip_weight",
        "distance_miles",
    ]
]

# filter to HBW
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="pCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "pTAZID"}
)
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="aCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "aTAZID"}
)

# fitler to WF subarea
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["pSUBAREAID"] == 1]
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["aSUBAREAID"] == 1]

# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_23.copy()
    .groupby("hh_id")["vehicle_id"]
    .count()
    .reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["hh_id", "vehown"]]

# income lookup table
income_lookup = hts_hh_23.copy()[["hh_id", "income_detailed"]]
income_lookup["income"] = np.select(
    [
        income_lookup["income_detailed"].isin([1, 2, 3, 4]),
        income_lookup["income_detailed"].isin([5, 6, 7, 8, 9, 10]),
    ],
    ["lo", "hi"],
    default="",
)

income_lookup = income_lookup[["hh_id", "income"]]

# merge vehown and income to trip table
hts_trip_23_merge = hts_trip_23_merge.merge(vehown_lookup, how="left", on="hh_id")
hts_trip_23_merge["vehown"] = hts_trip_23_merge["vehown"].fillna(0).astype(int)
hts_trip_23_merge = hts_trip_23_merge.merge(income_lookup, how="left", on="hh_id")

# calculate segment
hts_trip_23_merge["segment"] = np.where(
    hts_trip_23_merge["income"].notna() & (hts_trip_23_merge["income"] != ""),
    hts_trip_23_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_23_merge["income"].astype(str),
    "",
)

# filter to only known income trips
hts_trip_23_merge = hts_trip_23_merge[~(hts_trip_23_merge["segment"] == "")]

hts_trip_23_final = hts_trip_23_merge[
    ["vehown", "income", "pTAZID", "aTAZID", "trip_weight"]
]


## Model Data


### Retail, Industrial, and Other Employment PCT


In [70]:
if "N" in df_se.columns:
    df_se_indexed = df_se.set_index(df_se["N"] - 1)
else:
    df_se_indexed = df_se.copy()

df_se_full = df_se_indexed.reindex(np.arange(used_zones)).fillna(0)
if "N" in df_se_full.columns:
    df_se_full["N"] = np.arange(1, used_zones + 1)

# Employment Ratios
denom = (df_se_full["RETEMP"] + df_se_full["INDEMP"] + df_se_full["OTHEMP"]).values
retail_pct = np.divide(
    df_se_full["RETEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
ind_pct = np.divide(
    df_se_full["INDEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
other_pct = np.divide(
    df_se_full["OTHEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)

# Assuming df_se_full and your _pct arrays already exist from your snippet
df_se = pd.DataFrame(
    {
        "TAZID": df_se_full["N"]
        if "N" in df_se_full.columns
        else np.arange(1, used_zones + 1),
        "RETEMP": df_se_full["RETEMP"],
        "INDEMP": df_se_full["INDEMP"],
        "OTHEMP": df_se_full["OTHEMP"],
        "RETPCT": retail_pct,
        "INDPCT": ind_pct,
        "OTHPCT": other_pct,
    }
)

# Optional: Reset index if you want a clean 0-based integer index
df_se.reset_index(drop=True, inplace=True)

df_se["TOTEMP"] = df_se["RETEMP"] + df_se["INDEMP"] + df_se["OTHEMP"]
df_se


,TAZID,RETEMP,INDEMP,OTHEMP,RETPCT,INDPCT,OTHPCT,TOTEMP
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,2.0,0.0,0.0,1.0,2.0
...,...,...,...,...,...,...,...,...
3624,3625,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3625,3626,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3626,3627,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3627,3628,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Logsums by Segment


In [71]:
segments = ["0veh_lo", "0veh_hi", "1veh_lo", "1veh_hi", "2veh_lo", "2veh_hi"]
logsum_mtx = {}
with omx.open_file(path_logsums, "r") as f:
    for seg in segments:
        logsum_mtx[seg] = np.array(f[seg][:])


In [72]:
logsum_mtx["0veh_lo"]


array([[  2.64,   2.61,   0.11, ..., -21.95, -21.95, -21.95],
       [  2.61,   3.09,  -0.11, ..., -21.89, -21.89, -21.89],
       [  0.11,  -0.11,   2.59, ..., -21.13, -21.13, -21.13],
       ...,
       [ -0.77,  -0.77,  -0.77, ...,  -0.77,  -0.77,  -0.77],
       [ -0.77,  -0.77,  -0.77, ...,  -0.77,  -0.77,  -0.77],
       [ -0.77,  -0.77,  -0.77, ...,  -0.77,  -0.77,  -0.77]],
      shape=(3629, 3629))

### Highway Distance


In [73]:
with (
    omx.open_file(path_skm_hwy, "r") as skm_hwy,
    # omx.open_file(path_skm_walk, "r") as skm_walk,
):
    dist_mtx = np.array(skm_hwy["dist_GP"][:])
    # hwy_time = np.array(skm_hwy["ivt_GP"][:]) + np.array(skm_hwy["ovt"][:])
    # walk_gencost = np.array(skm_walk["GENCOST"][:])

dist_mtx


array([[  0.48,   0.5 ,  12.22, ..., 152.31, 132.78, 132.93],
       [  0.5 ,   0.23,  11.95, ..., 152.03, 132.5 , 132.65],
       [ 12.22,  11.95,   0.51, ..., 147.44, 127.91, 128.07],
       ...,
       [152.05, 151.77, 147.56, ...,   1.  ,  23.38,  23.52],
       [132.55, 132.27, 128.06, ...,  23.28,   1.  ,   2.2 ],
       [132.79, 132.51, 128.3 , ...,  23.52,   2.11,   1.  ]],
      shape=(3629, 3629))

### Skims


In [74]:
skims = logsum_mtx.copy()
skims["dist"] = dist_mtx

# 1. Determine dimensions (assuming N x N matrix)
n_rows, n_cols = skims["dist"].shape

# 2. Create the I and J columns (1-based indexing for TAZ IDs)
i_indices = np.repeat(np.arange(1, n_rows + 1), n_cols)
j_indices = np.tile(np.arange(1, n_cols + 1), n_rows)

# 3. Create a dictionary to hold the columns
data = {"I": i_indices, "J": j_indices}

# 4. Flatten each matrix in the skim dictionary and add it to the data
for key, matrix in skims.items():
    data[key] = matrix.flatten()

# 5. Create the final DataFrame
df_skim = pd.DataFrame(data)

# Optional: View the first few rows
df_skim


,I,J,0veh_lo,0veh_hi,1veh_lo,1veh_hi,2veh_lo,2veh_hi,dist
0,1,1,2.64,2.64,0.08,0.10,-0.09,-0.06,0.48
1,1,2,2.61,2.61,0.08,0.11,-0.08,-0.05,0.50
2,1,3,0.11,0.36,-2.17,-1.37,-2.31,-1.47,12.22
3,1,4,1.01,1.12,-2.04,-1.35,-2.28,-1.50,13.07
4,1,5,1.14,1.22,-0.38,-0.23,-0.47,-0.32,2.05
...,...,...,...,...,...,...,...,...,...
13169636,3629,3625,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,61.94
13169637,3629,3626,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,46.48
13169638,3629,3627,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,23.52
13169639,3629,3628,-0.77,-0.77,-0.77,-0.77,-0.77,-0.77,2.11


## Integrate Data


In [75]:
# -----------------------------
# LOAD DATA
# -----------------------------
trips = hts_trip_23_final.copy()
zones = df_se.copy()
zones = zones.rename(columns={"Z": "TAZID"})
skim = df_skim.copy()

# keep only zones with employment
zones = zones[zones["TOTEMP"] > 0].copy()

# -----------------------------
# CREATE ZONE LIST FOR SAMPLING
# -----------------------------
zones_ii = zones[zones["TAZID"] < 3563]
zone_ids = zones_ii["TAZID"].values


# -----------------------------
# SAMPLE FUNCTION
# -----------------------------
def sample_alternatives(row, n_sample=50):
    chosen = row["aTAZID"]

    # 1. Remove chosen from sampling pool
    pool = zone_ids[zone_ids != chosen]
    sampled = np.random.choice(pool, size=n_sample, replace=False)

    # 2. Include chosen alternative
    alts = np.append(sampled, chosen)

    # 3. Create a DataFrame from the ENTIRE row and repeat it 51 times
    # This preserves income, veh, trip_id, etc.
    df = pd.DataFrame([row] * len(alts))

    # 4. Overwrite the 'aTAZID' column with the sampled alternatives
    df["aTAZID"] = alts

    # 5. Mark choice
    df["CHOICE"] = (df["aTAZID"] == chosen).astype(int)

    return df


# -----------------------------
# BUILD CHOICE SET (The rest remains the same)
# -----------------------------
sampled_list = []
for _, row in trips.iterrows():
    sampled_list.append(sample_alternatives(row, 50))

choice_df = pd.concat(sampled_list, ignore_index=True)

# -----------------------------
# MERGE ZONAL DATA (destination side)
# -----------------------------
choice_df = choice_df.merge(zones, left_on="aTAZID", right_on="TAZID", how="left")

# -----------------------------
# MERGE SKIM
# -----------------------------
choice_df = choice_df.merge(
    skim, left_on=["pTAZID", "aTAZID"], right_on=["I", "J"], how="left"
)

# -----------------------------
# CREATE VARIABLES
# -----------------------------
# choice_df["dist"] = choice_df["dist"]
# choice_df["time"] = choice_df["TIME"]

choice_df["short_trip"] = 1 / np.maximum(1, choice_df["dist"])
choice_df["dist2"] = choice_df["dist"] ** 2
choice_df["dist3"] = choice_df["dist"] ** 3

choice_df["log_emp"] = np.log(choice_df["TOTEMP"])

# employment ratios
den = choice_df["RETEMP"] + choice_df["INDEMP"] + choice_df["OTHEMP"]

choice_df["retail_ratio"] = choice_df["RETEMP"] / den
choice_df["industrial_ratio"] = choice_df["INDEMP"] / den
choice_df["other_ratio"] = choice_df["OTHEMP"] / den

# intrazonal dummy
choice_df["intrazonal"] = (choice_df["pTAZID"] == choice_df["aTAZID"]).astype(int)

# unique observation id
choice_df["obs_id"] = np.repeat(np.arange(len(trips)), 51)

# -----------------------------
# SAVE FOR BIOGEME
# -----------------------------
choice_df


,vehown,income,pTAZID,aTAZID,trip_weight,CHOICE,TAZID,RETEMP,INDEMP,OTHEMP,...,dist,short_trip,dist2,dist3,log_emp,retail_ratio,industrial_ratio,other_ratio,intrazonal,obs_id
0,1,lo,2979.0,1507.0,17.687357,0,1507.0,307.0,54.0,844.0,...,42.68,0.023430,1821.5824,77745.136832,7.094235,0.254772,0.044813,0.700415,0,0
1,1,lo,2979.0,2767.0,17.687357,0,2767.0,138.0,63.0,473.0,...,13.10,0.076336,171.6100,2248.091000,6.513230,0.204748,0.093472,0.701780,0,0
2,1,lo,2979.0,403.0,17.687357,0,403.0,0.0,0.0,19.0,...,78.08,0.012807,6096.4864,476013.658112,2.944439,0.000000,0.000000,1.000000,0,0
3,1,lo,2979.0,552.0,17.687357,0,552.0,2.0,29.0,60.0,...,75.59,0.013229,5713.8481,431909.777879,4.510860,0.021978,0.318681,0.659341,0,0
4,1,lo,2979.0,1977.0,17.687357,0,1977.0,129.0,24.0,254.0,...,35.47,0.028193,1258.1209,44625.548323,6.008813,0.316953,0.058968,0.624079,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707926,2,hi,316.0,659.0,0.000000,0,659.0,40.0,17.0,475.0,...,15.94,0.062735,254.0836,4050.092584,6.276643,0.075188,0.031955,0.892857,0,13880
707927,2,hi,316.0,1518.0,0.000000,0,1518.0,1.0,1.0,49.0,...,50.96,0.019623,2596.9216,132339.124736,3.931826,0.019608,0.019608,0.960784,0,13880
707928,2,hi,316.0,1756.0,0.000000,0,1756.0,8.0,14.0,315.0,...,59.08,0.016926,3490.4464,206215.573312,5.820083,0.023739,0.041543,0.934718,0,13880
707929,2,hi,316.0,3178.0,0.000000,0,3178.0,1232.0,19.0,1015.0,...,92.78,0.010778,8608.1284,798662.152952,7.725771,0.543689,0.008385,0.447926,0,13880


In [76]:
# 1. Construct the name of the column we want to pick from
# (e.g., 1 + "veh_" + "lo" -> "1veh_lo")
choice_df["target_col"] = (
    choice_df["vehown"].astype(int).astype(str) + "veh_" + choice_df["income"]
)

# 2. Initialize the logsum column with zeros or NaNs
choice_df["logsum"] = np.nan

# 3. Efficiently pick the values using a loop over the unique combinations
# (This is much faster than row-by-row iteration)
for col_name in choice_df["target_col"].unique():
    mask = choice_df["target_col"] == col_name
    if col_name in choice_df.columns:
        choice_df.loc[mask, "logsum"] = choice_df.loc[mask, col_name]

# 4. Clean up the helper column
choice_df.drop(columns=["target_col"], inplace=True)

choice_df.to_csv("choice_data.csv", index=False)


## Estimate Coefficients & Constants


- distance values look good. 0 veh low income is twice as sensitive as 2 veh high income which is what we would expect
- intrazonal values are mostly positive, because why wouldn't you want to stay in the same zone if you could
- logsums between 0.2 and .4 (should be between 0 and 1.0) -- changes to network will appropriately affect work location
- dist2 and dist3 super close to original values
- short trip factor is positive, seems fine
- _RED FLAG_ - magnitude of employment ratios are super high -- recommend change is to drop one of the values (other) and rerun


In [ ]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("choice_data.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

varying_cols = [
    "dist",
    "dist2",
    "dist3",
    "short_trip",
    "log_emp",
    "logsum",
    "retail_ratio",
    "industrial_ratio",
    "other_ratio",
    "intrazonal",
]

# 🔥 include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
beta_short = Beta("beta_short", 0, None, None, 0)
beta_dist2 = Beta("beta_dist2", 0, None, None, 0)
beta_dist3 = Beta("beta_dist3", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


b_dist1 = create_segmented_beta("b_dist1", segments)
b_logsum = create_segmented_beta("b_logsum", segments)
b_iz = create_segmented_beta("b_iz", segments)
b_retail = create_segmented_beta("b_retail", segments)
b_ind = create_segmented_beta("b_ind", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 51 alternatives)
# ==================================================
V = {}
av = {}

for i in range(1, 52):
    V[i] = (
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        + beta_dist2 * Variable(f"dist2_{i}")
        + beta_dist3 * Variable(f"dist3_{i}")
        + beta_emp * Variable(f"log_emp_{i}")
        + b_logsum * Variable(f"logsum_{i}")
        + b_retail * Variable(f"retail_ratio_{i}")
        + b_ind * Variable(f"industrial_ratio_{i}")
        + b_iz * Variable(f"intrazonal_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# 🔥 define weight variable
WEIGHT = Variable("trip_weight")

# 🔥 apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

biogeme.modelName = "destination_choice_wide_weighted"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...
Original data shape: (707931, 33)


File biogeme.toml has been created


Estimated parameters:
                Value  Rob. Std err  Rob. t-test  Rob. p-value
b_dist1_H0  -0.128179      0.024976    -5.132160  2.864355e-07
b_dist1_H1  -0.069271      0.010936    -6.334352  2.383413e-10
b_dist1_H2  -0.098328      0.007888   -12.465071  0.000000e+00
b_dist1_L0  -0.211092      0.030155    -7.000144  2.557066e-12
b_dist1_L1  -0.070004      0.010128    -6.911898  4.782175e-12
b_dist1_L2  -0.082532      0.010896    -7.574594  3.597123e-14
b_ind_H0     0.472217      0.479565     0.984677  3.247829e-01
b_ind_H1    -0.425900      0.114306    -3.725967  1.945677e-04
b_ind_H2    -0.185828      0.063333    -2.934151  3.344619e-03
b_ind_L0     0.131864      0.404734     0.325804  7.445728e-01
b_ind_L1    -0.764431      0.170052    -4.495281  6.947825e-06
b_ind_L2    -0.053644      0.194877    -0.275273  7.831067e-01
b_iz_H0      1.397122      0.862843     1.619207  1.054027e-01
b_iz_H1      0.291097      0.350222     0.831179  4.058726e-01
b_iz_H2      2.329208      0.1970

For some reason, I cannont exactly replicate the results that andy got from his estimation process. We think it has to do with different version of the biogeme package. Even so, the above code replicates it to the best we can. The model will use the results of Andy's code however, which are pasted below:


In [ ]:
# Name     Value  Robust std err.  Robust t-stat.  Robust p-value
# 0    b_dist1_L0 -0.214188         0.030389       -7.048123    1.813438e-12
# 1    b_dist1_L1 -0.069419         0.010021       -6.927662    4.278577e-12
# 2    b_dist1_L2 -0.070592         0.008842       -7.983930    1.332268e-15
# 3    b_dist1_H0 -0.129514         0.025426       -5.093671    3.511954e-07
# 4    b_dist1_H1 -0.067517         0.010916       -6.184896    6.214347e-10
# 5    b_dist1_H2 -0.052757         0.007833       -6.735538    1.633249e-11
# 6   b_logsum_L0  0.317888         0.087772        3.621759    2.926071e-04
# 7   b_logsum_L1  0.559290         0.069902        8.001039    1.332268e-15
# 8   b_logsum_L2  0.533347         0.063921        8.343878    0.000000e+00
# 9   b_logsum_H0  0.153432         0.118704        1.292558    1.961641e-01
# 10  b_logsum_H1  0.779985         0.113790        6.854612    7.150502e-12
# 11  b_logsum_H2  1.011520         0.082361       12.281590    0.000000e+00
# 12  b_retail_L0  0.391739         0.421638        0.929089    3.528428e-01
# 13  b_retail_L1 -1.781417         0.251721       -7.076945    1.473710e-12
# 14  b_retail_L2 -0.965687         0.278864       -3.462925    5.343371e-04
# 15  b_retail_H0 -0.550355         0.687232       -0.800828    4.232315e-01
# 16  b_retail_H1 -2.465867         0.219801      -11.218626    0.000000e+00
# 17  b_retail_H2 -1.681870         0.104727      -16.059631    0.000000e+00
# 18     b_ind_L0 -0.072970         0.417019       -0.174980    8.610952e-01
# 19     b_ind_L1 -1.361789         0.200760       -6.783180    1.175593e-11
# 20     b_ind_L2 -0.892342         0.232661       -3.835371    1.253749e-04
# 21     b_ind_H0  0.050213         0.511104        0.098244    9.217383e-01
# 22     b_ind_H1 -0.637994         0.119038       -5.359601    8.340573e-08
# 23     b_ind_H2 -0.332155         0.065409       -5.078106    3.812163e-07
# 24      b_iz_L0  0.227189         0.725415        0.313184    7.541406e-01
# 25      b_iz_L1  0.546081         0.346851        1.574399    1.153953e-01
# 26      b_iz_L2  1.365067         0.581207        2.348676    1.884027e-02
# 27      b_iz_H0  0.562778         1.041309        0.540452    5.888852e-01
# 28      b_iz_H1  0.293047         0.353470        0.829058    4.070715e-01
# 29      b_iz_H2  2.357390         0.211035       11.170634    0.000000e+00


# Estimation Process 2


## Data Prep


### Employment Ratio by Job Sector


In [84]:
denom = (
    df_se_full["RETL"]
    + df_se_full["FOOD"]
    + df_se_full["MANU"]
    + df_se_full["WSLE"]
    + df_se_full["OFFI"]
    + df_se_full["GVED"]
    + df_se_full["HLTH"]
    + df_se_full["OTHR"]
).values


# 2. Helper function to handle safe division across all sectors
def calc_pct(col_name):
    return np.divide(
        df_se_full[col_name].values,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom != 0,
    )


# 3. Generate the percentage arrays
retl_pct = calc_pct("RETL")
food_pct = calc_pct("FOOD")
manu_pct = calc_pct("MANU")
wsle_pct = calc_pct("WSLE")
offi_pct = calc_pct("OFFI")
gved_pct = calc_pct("GVED")
hlth_pct = calc_pct("HLTH")
othr_pct = calc_pct("OTHR")

# 4. Construct the new DataFrame
df_se = pd.DataFrame(
    {
        "TAZID": df_se_full["N"]
        if "N" in df_se_full.columns
        else np.arange(1, used_zones + 1),
        # Raw Counts
        "RETL": df_se_full["RETL"],
        "FOOD": df_se_full["FOOD"],
        "MANU": df_se_full["MANU"],
        "WSLE": df_se_full["WSLE"],
        "OFFI": df_se_full["OFFI"],
        "GVED": df_se_full["GVED"],
        "HLTH": df_se_full["HLTH"],
        "OTHR": df_se_full["OTHR"],
        # Percentages
        "retl_ratio": retl_pct,
        "food_ratio": food_pct,
        "manu_ratio": manu_pct,
        "wsle_ratio": wsle_pct,
        "offi_ratio": offi_pct,
        "gved_ratio": gved_pct,
        "hlth_ratio": hlth_pct,
        "othr_ratio": othr_pct,
    }
)

df_se.reset_index(drop=True, inplace=True)

# 5. Calculate Total Employment for validation
df_se["TOTEMP"] = (
    df_se["RETL"]
    + df_se["FOOD"]
    + df_se["MANU"]
    + df_se["WSLE"]
    + df_se["OFFI"]
    + df_se["GVED"]
    + df_se["HLTH"]
    + df_se["OTHR"]
)
df_se


,TAZID,RETL,FOOD,MANU,WSLE,OFFI,GVED,HLTH,OTHR,retl_ratio,food_ratio,manu_ratio,wsle_ratio,offi_ratio,gved_ratio,hlth_ratio,othr_ratio,TOTEMP
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3624,3625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3625,3626,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3626,3627,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3627,3628,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [85]:
gdf_taz_copy = gdf_taz.copy()
df_se_copy = df_se.copy()

# sectors
sector_cols = ["RETL", "FOOD", "MANU", "WSLE", "OFFI", "GVED", "HLTH", "OTHR"]

# totemp recalc
df_se_copy["TOTEMP"] = df_se_copy[sector_cols].sum(axis=1)

# merge columns
merge_cols = ["TAZID", "TOTEMP"] + sector_cols

# Merge into the spatial dataframe copy
gdf_taz_se = gdf_taz_copy.merge(df_se_copy[merge_cols], on="TAZID", how="left")

# Project to UTM Zone 12N (Utah) if necessary
if gdf_taz_se.crs is None:
    gdf_taz_se = gdf_taz_se.set_crs(epsg=4326)
if gdf_taz_se.crs.is_geographic:
    gdf_taz_se = gdf_taz_se.to_crs(epsg=26912)

# 2. Build the KNN weights (k=8 nearest neighbors)
weights = KNN.from_dataframe(gdf_taz_se, k=8)
neighbor_dict = weights.neighbors

# 3. Create a list of columns to calculate density for (Total + Sectors)
cols_to_calculate = ["TOTEMP"] + sector_cols

# 4. Calculate the smoothed values
job_densities = []

for idx, neighbors in neighbor_dict.items():
    # Combine the focal zone index with its 8 neighbors (Total = 9)
    group_indices = neighbors + [idx]
    group_gdf = gdf_taz_se.loc[group_indices]

    # Calculate Total Area in the cluster (sq miles)
    total_area = group_gdf.geometry.area.sum() / 2589988.0

    # Start the dictionary for this row
    taz_results = {"TAZID": gdf_taz_se.loc[idx, "TAZID"]}

    # Loop through TOTEMP and all individual sectors
    for col in cols_to_calculate:
        total_col_emp = group_gdf[col].sum()
        density = total_col_emp / total_area if total_area > 0 else 0
        taz_results[f"{col}_density"] = density

    job_densities.append(taz_results)

# 5. Merge the results back into our working spatial dataframe
results_df = pd.DataFrame(job_densities)
gdf_taz_se = gdf_taz_se.merge(results_df, on="TAZID")

# 6. View the final result cleanly
display_cols = ["TAZID", "TOTEMP_density"] + [f"{col}_density" for col in sector_cols]
taz_job_dens = gdf_taz_se[display_cols]
taz_job_dens


,TAZID,TOTEMP_density,RETL_density,FOOD_density,MANU_density,WSLE_density,OFFI_density,GVED_density,HLTH_density,OTHR_density
0,1.0,28.568452,0.000000,0.000000,8.484908,19.227269,0.000000,0.000000,0.155686,0.700589
1,2.0,24.952125,1.289320,0.000000,8.266813,11.679718,2.730324,0.000000,0.151685,0.834266
2,3.0,36.520922,0.148459,0.000000,8.091017,27.019544,0.000000,0.000000,0.148459,1.113443
3,4.0,30.921992,0.153841,0.000000,8.384321,21.076183,0.000000,0.000000,0.153841,1.153806
4,5.0,24.952125,1.289320,0.000000,8.266813,11.679718,2.730324,0.000000,0.151685,0.834266
...,...,...,...,...,...,...,...,...,...,...
3557,3127.0,60.568894,9.911274,0.275313,1.651879,1.101253,6.607516,7.158142,1.927192,31.936326
3558,3244.0,203.438163,35.495721,3.178721,3.178721,0.000000,18.542541,25.429770,17.482967,100.129721
3559,2369.0,60.261677,0.000000,0.379004,0.379004,0.000000,20.466230,37.142417,1.895021,0.000000
3560,2373.0,112.236392,2.416573,21.212141,0.268508,1.611049,24.702747,29.535893,13.425406,19.064076


### Intersection Density


In [79]:
df_int = df_dens.copy()
df_int = df_int[["TAZID", "IntDen"]]


### Parking Cost and CBD


In [80]:
df_taz_cbd_prk = df_taz.copy()
df_taz_cbd_prk = df_taz_cbd_prk[["TAZID", "CBD", "PRKCSTPERM", "PRKCSTTEMP"]]


### Transit Time


In [81]:
# The same helper function from before remains exactly the same
import numpy as np


def get_min_valid_time(matrix_list):
    stacked = np.stack(matrix_list)
    stacked_no_zeros = np.where(stacked == 0, np.inf, stacked)
    min_time = np.min(stacked_no_zeros, axis=0)
    return np.where(np.isinf(min_time), 0, min_time)


with (
    omx.open_file(path_skm_transit_pk, "r") as skm_transit_pk,
    omx.open_file(path_skm_transit_ok, "r") as skm_transit_ok,
):
    # 1. Local Transit (Mode 4) - Pulling from BOTH peak and off-peak
    local_keys = ["w4time", "d4time"]
    local_matrices = [np.array(skm_transit_pk[k][:]) for k in local_keys] + [
        np.array(skm_transit_ok[k][:]) for k in local_keys
    ]

    # 2. Premium Transit (Modes 5 through 9) - Pulling from BOTH peak and off-peak
    premium_keys = [
        "w5time",
        "w6time",
        "w7time",
        "w8time",
        "w9time",
        "d5time",
        "d6time",
        "d7time",
        "d8time",
        "d9time",
    ]
    premium_matrices = [np.array(skm_transit_pk[k][:]) for k in premium_keys] + [
        np.array(skm_transit_ok[k][:]) for k in premium_keys
    ]

    # 3. Calculate the absolute shortest valid times across the whole day
    local_min_time_daily = get_min_valid_time(local_matrices)
    premium_min_time_daily = get_min_valid_time(premium_matrices)


In [82]:
# 1. Determine dimensions (assuming the zone system is a square N x N matrix)
n_rows, n_cols = local_min_time_daily.shape

# 2. Create the I and J columns (1-based indexing for TAZ IDs)
i_indices = np.repeat(np.arange(1, n_rows + 1), n_cols)
j_indices = np.tile(np.arange(1, n_cols + 1), n_rows)

# 3. Flatten the matrices and create the final DataFrame directly
df_transit_skims = pd.DataFrame(
    {
        "I": i_indices,
        "J": j_indices,
        "lcl_transit_cost": local_min_time_daily.flatten(),
        "prm_transit_cost": premium_min_time_daily.flatten(),
    }
)


## Integrate Data


In [ ]:
# -----------------------------
# LOAD DATA
# -----------------------------
# trip data
trips = hts_trip_23_final.copy()

# zones level data
zones = df_se.copy()
zones = zones.rename(columns={"Z": "TAZID"})
zones = zones.merge(taz_job_dens, on="TAZID", how="left")
zones = zones.merge(df_int, on="TAZID", how="left")
zones = zones.merge(df_taz_cbd_prk, on="TAZID", how="left")

# skim data
dskim = df_skim.copy()
tskim = df_transit_skims.copy()
skim = dskim.merge(tskim, left_on=["I", "J"], right_on=["I", "J"], how="inner")

# keep only zones with employment
zones = zones[zones["TOTEMP"] > 0].copy()

# -----------------------------
# CREATE ZONE LIST FOR SAMPLING
# -----------------------------
zones_ii = zones[zones["TAZID"] < 3563]
zone_ids = zones_ii["TAZID"].values


# -----------------------------
# SAMPLE FUNCTION
# -----------------------------
def sample_alternatives(row, n_sample=50):
    chosen = row["aTAZID"]

    # 1. Remove chosen from sampling pool
    pool = zone_ids[zone_ids != chosen]
    sampled = np.random.choice(pool, size=n_sample, replace=False)

    # 2. Include chosen alternative
    alts = np.append(sampled, chosen)

    # 3. Create a DataFrame from the ENTIRE row and repeat it 51 times
    # This preserves income, veh, trip_id, etc.
    df = pd.DataFrame([row] * len(alts))

    # 4. Overwrite the 'aTAZID' column with the sampled alternatives
    df["aTAZID"] = alts

    # 5. Mark choice
    df["CHOICE"] = (df["aTAZID"] == chosen).astype(int)

    return df


# -----------------------------
# BUILD CHOICE SET (The rest remains the same)
# -----------------------------
sampled_list = []
for _, row in trips.iterrows():
    sampled_list.append(sample_alternatives(row, 50))

choice_df = pd.concat(sampled_list, ignore_index=True)

# -----------------------------
# MERGE ZONAL DATA (destination side)
# -----------------------------
choice_df = choice_df.merge(zones, left_on="aTAZID", right_on="TAZID", how="left")

# -----------------------------
# MERGE SKIM
# -----------------------------
choice_df = choice_df.merge(
    skim, left_on=["pTAZID", "aTAZID"], right_on=["I", "J"], how="left"
)

# -----------------------------
# CREATE VARIABLES
# -----------------------------

choice_df["short_trip"] = 1 / np.maximum(1, choice_df["dist"])
choice_df["dist2"] = choice_df["dist"] ** 2
choice_df["dist3"] = choice_df["dist"] ** 3

choice_df["log_emp"] = np.log(choice_df["TOTEMP"])

# intrazonal dummy
choice_df["intrazonal"] = (choice_df["pTAZID"] == choice_df["aTAZID"]).astype(int)

# unique observation id
choice_df["obs_id"] = np.repeat(np.arange(len(trips)), 51)

# -----------------------------
# SAVE FOR BIOGEME
# -----------------------------
# choice_df.to_csv("choice_data_full.csv", index=False)


,vehown,income,pTAZID,aTAZID,trip_weight,CHOICE,TAZID,RETL,FOOD,MANU,...,2veh_hi,dist,lcl_transit_cost,prm_transit_cost,short_trip,dist2,dist3,log_emp,intrazonal,obs_id
0,1,lo,2979.0,939.0,17.687357,0,939.0,42.0,12.0,0.0,...,-5.39,51.36,0.00,0.00,0.019470,2637.8496,1.354800e+05,7.044905,0,0
1,1,lo,2979.0,85.0,17.687357,0,85.0,3.0,0.0,0.0,...,-9.64,100.49,0.00,0.00,0.009951,10098.2401,1.014772e+06,4.406719,0,0
2,1,lo,2979.0,1126.0,17.687357,0,1126.0,66.0,102.0,0.0,...,-4.65,44.52,0.00,97.96,0.022462,1982.0304,8.823999e+04,6.629363,0,0
3,1,lo,2979.0,22.0,17.687357,0,22.0,0.0,0.0,9.0,...,-9.85,103.58,0.00,0.00,0.009654,10728.8164,1.111291e+06,2.197225,0,0
4,1,lo,2979.0,2281.0,17.687357,0,2281.0,0.0,1.0,1.0,...,-3.69,33.25,0.00,0.00,0.030075,1105.5625,3.675995e+04,3.828641,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707926,2,hi,316.0,2691.0,0.000000,0,2691.0,15.0,2.0,2.0,...,-7.17,72.41,0.00,0.00,0.013810,5243.2081,3.796607e+05,4.564348,0,13880
707927,2,hi,316.0,346.0,0.000000,0,346.0,2.0,47.0,1.0,...,-0.44,2.65,32.30,0.00,0.377358,7.0225,1.860962e+01,4.844187,0,13880
707928,2,hi,316.0,3036.0,0.000000,0,3036.0,46.0,16.0,94.0,...,-8.39,87.71,0.00,160.06,0.011401,7693.0441,6.747569e+05,6.788972,0,13880
707929,2,hi,316.0,2189.0,0.000000,0,2189.0,0.0,0.0,0.0,...,-5.49,50.43,0.00,0.00,0.019829,2543.1849,1.282528e+05,2.302585,0,13880


## Estimate with New Values


- [x] job density --> taz level by employment sector, employment of 8 nearest zones (total 9) divided by area of those zones
- [x] logsum --> (already there)
- [x] transit time --> mode output mc skim, premium and local, matrix skims of times (for premium choose the best time for each od pair premium mode)
- [x] parking cost --> taz shapefile
- [x] cbd dummy --> taz shapefile
- [x] intersection density --> processing in input processing at taz level
- [x] ratio for all job sectors

Last Resort

- district-pair constants -> flag i-j pairs iwth value of 1 if they need something special
  - 21-16, 16-16, 14-14, 14-13,


In [ ]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("choice_data.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

varying_cols = [
    "dist",
    "dist2",
    "dist3",
    "short_trip",
    "log_emp",
    "logsum",
    "retail_ratio",
    "industrial_ratio",
    "other_ratio",
    "intrazonal",
]

# 🔥 include trip_weight here
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

segments = {
    "L0": (INCGRP == 0) * (VEHGRP == 0),
    "L1": (INCGRP == 0) * (VEHGRP == 1),
    "L2": (INCGRP == 0) * (VEHGRP == 2),
    "H0": (INCGRP == 1) * (VEHGRP == 0),
    "H1": (INCGRP == 1) * (VEHGRP == 1),
    "H2": (INCGRP == 1) * (VEHGRP == 2),
}

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
beta_short = Beta("beta_short", 0, None, None, 0)
beta_dist2 = Beta("beta_dist2", 0, None, None, 0)
beta_dist3 = Beta("beta_dist3", 0, None, None, 0)
beta_emp = Beta("beta_emp", 0, None, None, 0)


def create_segmented_beta(name, seg_dict):
    expr = 0
    for s_name, mask in seg_dict.items():
        b = Beta(f"{name}_{s_name}", 0, None, None, 0)
        expr += b * mask
    return expr


b_dist1 = create_segmented_beta("b_dist1", segments)
b_logsum = create_segmented_beta("b_logsum", segments)
b_iz = create_segmented_beta("b_iz", segments)
b_retail = create_segmented_beta("b_retail", segments)
b_ind = create_segmented_beta("b_ind", segments)

# ==================================================
# 5. UTILITY DICTIONARY (For 51 alternatives)
# ==================================================
V = {}
av = {}

for i in range(1, 52):
    V[i] = (
        beta_short * Variable(f"short_trip_{i}")
        + b_dist1 * Variable(f"dist_{i}")
        + beta_dist2 * Variable(f"dist2_{i}")
        + beta_dist3 * Variable(f"dist3_{i}")
        + beta_emp * Variable(f"log_emp_{i}")
        + b_logsum * Variable(f"logsum_{i}")
        + b_retail * Variable(f"retail_ratio_{i}")
        + b_ind * Variable(f"industrial_ratio_{i}")
        + b_iz * Variable(f"intrazonal_{i}")
    )
    av[i] = 1

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# 🔥 define weight variable
WEIGHT = Variable("trip_weight")

# 🔥 apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

biogeme.modelName = "destination_choice_wide_weighted"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())
